# Mini CBN — le moteur de proposition en une fonction

Version **réduite et lisible** de la règle de proposition CBN de *Ma Routine Appro*, pour la lire, la rejouer à la main ou la discuter.
Elle produit **exactement** les quantités et les dates du moteur de l'application dans ce cadre simplifié (un test automatique, `backend/tests/test_mini_cbn.py`, le vérifie à chaque livraison) :

* un article, **un fournisseur** ; jours ouvrés `S1.1` (lundi de la semaine 1) … `S10.5` (vendredi de la semaine 10), sans fermeture ;
* un **besoin par jour ouvré** ; un **stock en fin de veille** d'aujourd'hui ;
* pas d'ordre ferme, pas de quantité à retenir, pas d'ajustement, pas de refus, pas de plafond ni de quota ;
* un seul calcul par appel : les propositions (quantité, jour de livraison, date limite, marques) et la projection jour par jour.

Règles reproduites (`docs/regles_metier.md` § 7) : stock cible = besoin des *n* prochains jours ouvrés ; déclenchement sous (100 − tolérance) % de la cible ou sur une rupture ; retour à 100 % de la cible, MOQ puis multiple d'UM ; livraison sur le **dernier jour standard** avant le besoin ; **date limite** = livraison − (délai − marge) ; **tenable** ⇔ date limite ≥ aujourd'hui ; sinon attente de la livraison standard suivante si rien ne manque avant, **hors jour standard ◇** au premier jour ouvré tenable (sauf jours stricts), ou **plancher !** au jour du besoin ; le besoin non servi est **perdu** (le stock ne descend pas sous 0).


In [ ]:
import math
import pandas as pd

EPS = 1e-6


def jours(semaines=10):
    """Les jours ouvrés S1.1 … S10.5 (semaine.jour, 1 = lundi … 5 = vendredi)."""
    return [f"S{s}.{d}" for s in range(1, semaines + 1) for d in range(1, 6)]


def etiquette(i):
    """Index de jour ouvré → étiquette (S0.x pour une date limite avant S1.1)."""
    return f"S{i // 5 + 1}.{i % 5 + 1}"


def projeter(stock_depart, besoin, entrees):
    """Stock projeté jour par jour (fin de journée) : un besoin non servi est perdu (le stock ne descend pas sous 0)."""
    stock, manque, s = [], [], stock_depart
    for b, e in zip(besoin, entrees):
        solde = s + e - b
        manque.append(max(0.0, -solde))
        s = max(0.0, solde)
        stock.append(s)
    return stock, manque


def cbn(besoin, stock, aujourd_hui, couverture, tolerance, delai, marge, jours_standard, jours_stricts, moq, um, semaines=10):
    """Propositions CBN d'un article mono-fournisseur, sans ordre ferme, sans refus, sans plafond.

    besoin        : {"S1.1": 100, …} quantité consommée chaque jour ouvré (absent = 0)
    stock         : stock en fin de veille d'aujourd'hui
    aujourd_hui   : "S1.1" … (jour ouvré)
    couverture    : stock cible = besoin des n prochains jours ouvrés
    tolerance     : % sous la cible toléré avant de proposer un retour à la cible
    delai, marge  : délai fournisseur et marge de délai, en jours ouvrés ; date limite = livraison − (délai − marge)
    jours_standard: jours de livraison standard du fournisseur, ex. {1} = le lundi, {1,2,3,4,5} = tous les jours
    jours_stricts : True = jamais de livraison hors jour standard (◇)
    moq, um       : quantité minimale et multiple de commande
    """
    J = jours(semaines)
    n, t = len(J), J.index(aujourd_hui)
    b = [float(besoin.get(j, 0)) if i >= t else 0.0 for i, j in enumerate(J)]   # le stock est celui de la veille : le passé ne consomme plus
    cible = [sum(b[i + 1:i + 1 + couverture]) for i in range(n)]
    seuil = [c * (1 - tolerance / 100) for c in cible]
    tenable = t + max(delai - marge, 0)          # première livraison dont la commande peut encore partir aujourd'hui
    std = lambda i: i % 5 + 1 in jours_standard  # i est un jour de livraison standard ?
    demain = t + 1
    entrees = [0.0] * n
    props = {}                                   # jour de livraison → proposition
    stock_proj, manque = projeter(stock, b, entrees)

    def jour_livraison(i):
        """Jour de livraison du besoin du jour i → (jour, hors jour standard, urgent) ou None."""
        jour = i
        while jour < n:
            s = next((d for d in range(jour, demain - 1, -1) if std(d)), None)       # dernier jour standard ≤ jour
            if s is not None and s >= tenable:                                       # 1. jour standard tenable
                return s, False, False
            nxt = next((d for d in range(jour + 1, n) if std(d) and d >= tenable), None)
            if nxt is not None and not any(m > EPS for m in manque[i:nxt]):          # 2. rien ne manque avant : on attend
                return nxt, False, False
            if not jours_stricts:
                d = max(tenable, demain)
                if d <= jour:                                                        # 3. ◇ premier jour tenable avant le besoin
                    return d, not std(d), False
                return jour, not std(jour), True                                     # 4. plancher au jour du besoin
            if s is not None:                                                        # 4 bis. jours stricts : plancher au jour standard
                return s, False, True
            jour += 1                                                                # 5. pas de jour standard avant : glisse
        return None

    i = demain
    while i < n:
        court, sous = manque[i] > EPS, stock_proj[i] < seuil[i] - EPS
        if not (court or sous):
            i += 1
            continue
        r = jour_livraison(i)
        if r is None:
            break
        j, hors, urgent = r
        if urgent and not court:                 # dans le délai, seule une rupture vaut une commande urgente
            i += 1
            continue
        k = max(i, j)                            # le besoin s'évalue au jour de la livraison quand elle attend
        if urgent:
            if manque[k] <= EPS:
                i = k + 1
                continue
            net = manque[k]
        else:
            if stock_proj[k] >= seuil[k] - EPS and manque[k] <= EPS:
                i = k + 1
                continue
            net = cible[k] - stock_proj[k] + manque[k]
        qte = math.ceil(max(net, moq) / um - 1e-9) * um
        avant = stock_proj[k]
        entrees[j] += qte
        stock_proj, manque = projeter(stock, b, entrees)
        p = props.setdefault(j, dict(livraison=J[j], quantite=0.0, date_limite=etiquette(j - max(delai - marge, 0)), urgent=False,
                                     plancher=True, hors_jour_standard=False, besoin_net=0.0, stock_avant=avant, stock_apres=0.0, motif=""))
        p.update(quantite=p["quantite"] + qte, besoin_net=p["besoin_net"] + net, urgent=p["urgent"] or urgent, plancher=p["plancher"] and urgent,
                 hors_jour_standard=p["hors_jour_standard"] or hors, stock_apres=stock_proj[k])
        motif = (f"plancher : rupture de {net:g} le {J[k]}, aucune livraison tenable avant" if urgent
                 else f"stock {avant:g} < {100 - tolerance:g} % de la cible ({seuil[k]:g}) le {J[k]} : retour à la cible {cible[k]:g}")
        if hors:
            motif += " ; ◇ hors jour standard"
        if j > i:
            motif += f" (sous la cible dès {J[i]}, livraison standard suivante {J[j]})"
        p["motif"] = motif if not p["motif"] else (p["motif"] if p["motif"].count(" | ") >= 2 else f"{p['motif']} | {motif}")
        i = k + 1

    propositions = pd.DataFrame([props[j] for j in sorted(props)])
    projection = pd.DataFrame(dict(jour=J, besoin=b, cbn=entrees, stock=stock_proj, manque=manque, cible=cible, seuil=seuil))
    return propositions, projection


In [ ]:
# ---------------- Données d'entrée
semaines = 10                      # S1.1 … S10.5
aujourd_hui = "S1.3"               # mercredi de la semaine 1
stock = 250                        # stock en fin de veille

# besoin par jour ouvré : 100 par jour du lundi au vendredi (modifier librement, ex. besoin["S4.2"] = 300)
besoin = {f"S{s}.{d}": 100 for s in range(1, semaines + 1) for d in range(1, 6)}

couverture = 5                     # stock cible = besoin des 5 prochains jours ouvrés
tolerance = 15                     # % sous la cible toléré (règle commune, défaut 15)
delai = 4                          # délai fournisseur, jours ouvrés
marge = 1                          # marge de délai, jours ouvrés (règle commune, défaut 1 ; 0 = strict)
jours_standard = {2, 4}            # livraisons standard le mardi et le jeudi (1 = lundi … 5 = vendredi)
jours_stricts = False              # True : jamais de livraison hors jour standard (◇)
moq, um = 100, 50                  # quantité minimale, multiple de commande


In [ ]:
propositions, projection = cbn(besoin, stock, aujourd_hui, couverture, tolerance, delai, marge, jours_standard, jours_stricts, moq, um, semaines)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)
print(propositions.drop(columns=["motif"]).to_string(index=False))
print()
for m in propositions["motif"]: print("·", m)


livraison  quantite date_limite  urgent  plancher  hors_jour_standard  besoin_net  stock_avant  stock_apres
     S1.5     100.0        S1.2    True      True                True        50.0          0.0         50.0
     S2.1     550.0        S1.3   False     False                True       550.0          0.0        500.0
     S2.2     200.0        S1.4   False     False               False       200.0        400.0        500.0
     S2.4     300.0        S2.1   False     False               False       300.0        400.0        500.0
     S3.2     200.0        S2.4   False     False               False       200.0        400.0        500.0
     S3.4     300.0        S3.1   False     False               False       300.0        400.0        500.0
     S4.2     200.0        S3.4   False     False               False       200.0        400.0        500.0
     S4.4     300.0        S4.1   False     False               False       300.0        400.0        500.0
     S5.2     200.0        S

In [ ]:
# projection jour par jour (stock en fin de journée, manque = besoin non servi, perdu)
print(projection.iloc[jours(semaines).index(aujourd_hui):].head(25).to_string(index=False))


jour  besoin   cbn  stock  manque  cible  seuil
S1.3   100.0   0.0  150.0     0.0  500.0  425.0
S1.4   100.0   0.0   50.0     0.0  500.0  425.0
S1.5   100.0 100.0   50.0     0.0  500.0  425.0
S2.1   100.0 550.0  500.0     0.0  500.0  425.0
S2.2   100.0 200.0  600.0     0.0  500.0  425.0
S2.3   100.0   0.0  500.0     0.0  500.0  425.0
S2.4   100.0 300.0  700.0     0.0  500.0  425.0
S2.5   100.0   0.0  600.0     0.0  500.0  425.0
S3.1   100.0   0.0  500.0     0.0  500.0  425.0
S3.2   100.0 200.0  600.0     0.0  500.0  425.0
S3.3   100.0   0.0  500.0     0.0  500.0  425.0
S3.4   100.0 300.0  700.0     0.0  500.0  425.0
S3.5   100.0   0.0  600.0     0.0  500.0  425.0
S4.1   100.0   0.0  500.0     0.0  500.0  425.0
S4.2   100.0 200.0  600.0     0.0  500.0  425.0
S4.3   100.0   0.0  500.0     0.0  500.0  425.0
S4.4   100.0 300.0  700.0     0.0  500.0  425.0
S4.5   100.0   0.0  600.0     0.0  500.0  425.0
S5.1   100.0   0.0  500.0     0.0  500.0  425.0
S5.2   100.0 200.0  600.0     0.0  500.0